# LeetCode 3444 — Minimum Increments for Target Multiples in an Array bằng Python

- Link: https://leetcode.com/problems/minimum-increments-for-target-multiples-in-an-array/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động bitmask** trên tập `target` (≤ 4 phần tử), **BCNN** của tập con, duyệt **tập con của phần bù**, **cắt tỉa ứng viên**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumIncrements(self, nums: List[int], target: List[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra: một phần tử `x` của `nums` (sau khi tăng) có thể "phục vụ" **một nhóm** target cùng lúc ⇔ nó là bội của **BCNN** của nhóm đó.
2. Chi phí đưa `x` lên bội nhỏ nhất ≥ `x` của `L`: `(−x) mod L`.
3. DP bitmask: `dp[mask]` = chi phí nhỏ nhất để phủ tập target `mask` bằng các phần tử đã xét; mỗi phần tử chọn một tập con của phần **chưa phủ**.
4. Cắt tỉa: với mỗi tập con chỉ cần giữ `t` phần tử rẻ nhất ⇒ DP trên ≤ `15·4` ứng viên.
5. Cách nhìn khác: phân hoạch target thành nhóm (số Bell `B(4) = 15`) + ghép mỗi nhóm với một phần tử khác nhau.

## 2. Đề bài & ví dụ

Mỗi thao tác tăng một phần tử của `nums` thêm 1. Tìm số thao tác ít nhất để **mỗi** phần tử của `target` có **ít nhất một bội** trong `nums`.

**Ví dụ 1:** `nums = [1,2,3], target = [4]` → `1`. **Ví dụ 2:** `nums = [8,4], target = [10,5]` → `2` (8 → 10 phục vụ cả 10 và 5). **Ví dụ 3:** `nums = [7,9,10], target = [7]` → `0`.

**Ràng buộc:** `n ≤ 5·10^4`, `1 ≤ |target| ≤ 4 ≤ …`, `|target| ≤ n`, giá trị ≤ `10^4`.

In [1]:
from typing import List
from math import lcm

examples = [([1, 2, 3], [4], 1), ([8, 4], [10, 5], 2), ([7, 9, 10], [7], 0)]
for nums, target, exp in examples:
    print(nums, target, '-> kỳ vọng', exp)

[1, 2, 3] [4] -> kỳ vọng 1
[8, 4] [10, 5] -> kỳ vọng 2
[7, 9, 10] [7] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Chi phí lên bội gần nhất

Bội nhỏ nhất của `L` mà `≥ x` là `⌈x / L⌉ · L`; chi phí `⌈x/L⌉·L − x = (−x) mod L` (Python cho số dư không âm).

In [2]:
def cost(x, L):
    return (-x) % L

print(cost(3, 4), cost(8, 10), cost(7, 7), cost(9, 7))

1 2 0 5


### 3.2. Bội chung ⇔ bội của BCNN

`v` chia hết cho mọi phần tử của nhóm `G` ⇔ `v` chia hết cho `lcm(G)`. Với 4 số ≤ `10^4`, BCNN có thể tới ~`10^{16}` — Python xử lý số lớn thoải mái.

In [3]:
def subset_lcms(target):
    t = len(target)
    L = [1] * (1 << t)
    for m in range(1, 1 << t):
        low = (m & -m).bit_length() - 1
        L[m] = lcm(L[m ^ (1 << low)], target[low])
    return L

print(subset_lcms([10, 5]), subset_lcms([4, 6, 9]))

[1, 10, 5, 10] [1, 4, 6, 12, 9, 36, 18, 36]


### 3.3. Duyệt mọi tập con của một mặt nạ

`sub = rest; while sub: …; sub = (sub − 1) & rest`. Tổng số cặp `(mask, sub ⊆ phần bù)` là `3^t`.

In [4]:
rest = 0b1011
subs, sub = [], rest
while sub:
    subs.append(bin(sub)); sub = (sub - 1) & rest
print(subs)

['0b1011', '0b1010', '0b1001', '0b1000', '0b11', '0b10', '0b1']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `target = [10, 5]`.

- Phục vụ riêng: 10 cần `8 → 10` (2) hoặc `4 → 10` (6); 5 cần `4 → 5` (1) hoặc `8 → 10` (2). Riêng rẽ tốt nhất dùng **hai phần tử khác nhau**: `8 → 10` (cho 10) + `4 → 5` (cho 5) = 3.
- Phục vụ chung bằng một phần tử: `lcm(10, 5) = 10`: `8 → 10` = 2 ⇒ **2**.

⇒ Phải xét việc gộp nhóm target vào cùng một phần tử.

In [5]:
print('riêng:', cost(8, 10) + cost(4, 5), '| chung bằng 8:', cost(8, lcm(10, 5)), '| chung bằng 4:', cost(4, 10))

riêng: 3 | chung bằng 8: 2 | chung bằng 4: 6


## 5. Cách 1 — Brute Force: gán mỗi target cho một phần tử

In [6]:
import itertools

def min_inc_brute(nums, target):
    best = None
    for assign in itertools.product(range(len(nums)), repeat=len(target)):
        groups = {}
        for j, i in enumerate(assign):
            groups.setdefault(i, []).append(target[j])
        c = sum(cost(nums[i], lcm(*g)) for i, g in groups.items())
        if best is None or c < best:
            best = c
    return best

for nums, target, exp in examples:
    print(min_inc_brute(nums, target), '| kỳ vọng', exp)

1 | kỳ vọng 1
2 | kỳ vọng 2
0 | kỳ vọng 0


**Độ phức tạp:** `O(n^t · t)` — chỉ dùng cho `n` nhỏ.

Vì sao gán là đủ: trong lời giải tối ưu, mỗi target được phục vụ bởi một phần tử nào đó; phần tử nhận nhóm `G` phải là bội của `lcm(G)`, và tăng nó tới bội nhỏ nhất là tối ưu; phần tử không nhận target nào thì không tăng.

## 6. Ý tưởng thuật toán / bước đột phá

**DP bitmask theo từng phần tử:** `dp[mask]` = chi phí nhỏ nhất để phủ `mask` bằng các phần tử đã xét. Với phần tử `x`: hoặc bỏ qua, hoặc chọn tập con `sub` của phần chưa phủ, cộng `cost(x, L[sub])`:

`new[mask | sub] = min(new[mask | sub], dp[mask] + cost(x, L[sub]))`.

Mỗi phần tử `O(3^t) = 81` phép ⇒ `O(81 n)`.

**Cắt tỉa:** trong lời giải tối ưu có ≤ `t` phần tử được dùng; phần tử dùng cho nhóm `sub` có thể thay bằng một trong `t` phần tử rẻ nhất cho `sub` (trong `t` cái đó luôn có một cái chưa bị nhóm khác dùng). ⇒ Chỉ cần giữ `t` phần tử rẻ nhất cho mỗi `sub` (≤ `15 · 4 = 60` ứng viên), rồi chạy DP trên chúng.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP bitmask trên toàn bộ `nums`

In [7]:
def min_inc_dp(nums, target):
    t = len(target)
    full = (1 << t) - 1
    L = subset_lcms(target)
    INF = float('inf')
    dp = [INF] * (1 << t)
    dp[0] = 0
    for x in nums:
        c = [cost(x, L[m]) for m in range(1 << t)]
        new = dp[:]
        for mask in range(1 << t):
            if dp[mask] == INF:
                continue
            rest = full ^ mask
            sub = rest
            while sub:
                v = dp[mask] + c[sub]
                if v < new[mask | sub]:
                    new[mask | sub] = v
                sub = (sub - 1) & rest
        dp = new
    return dp[full]

for nums, target, exp in examples:
    print(min_inc_dp(nums, target), '| kỳ vọng', exp)

1 | kỳ vọng 1
2 | kỳ vọng 2
0 | kỳ vọng 0


### 7.2. Cách 3 — Cắt tỉa ứng viên + DP (lời giải chính)

In [8]:
import heapq

def candidates(nums, target):
    t = len(target)
    L = subset_lcms(target)
    keep = set()
    for m in range(1, 1 << t):
        for _, i in heapq.nsmallest(t, ((cost(x, L[m]), i) for i, x in enumerate(nums))):
            keep.add(i)
    return sorted(keep)

def min_inc_pruned(nums, target):
    cand = candidates(nums, target)
    return min_inc_dp([nums[i] for i in cand], target)

for nums, target, exp in examples:
    print(min_inc_pruned(nums, target), '| kỳ vọng', exp, '| số ứng viên:', len(candidates(nums, target)))

1 | kỳ vọng 1 | số ứng viên: 1
2 | kỳ vọng 2 | số ứng viên: 2
0 | kỳ vọng 0 | số ứng viên: 1


### 7.3. Cách 4 — Phân hoạch target thành nhóm + ghép với phần tử khác nhau

Liệt kê mọi **phân hoạch** của `t` target (≤ 15 cách với `t = 4`). Mỗi nhóm cần một phần tử **riêng**; với mỗi nhóm chỉ cần xét `t` phần tử rẻ nhất, rồi thử mọi cách chọn phần tử khác nhau (≤ `4^4` tổ hợp).

In [9]:
def set_partitions(items):
    if not items:
        yield []
        return
    first, rest = items[0], items[1:]
    for part in set_partitions(rest):
        yield [[first]] + part
        for i in range(len(part)):
            yield part[:i] + [[first] + part[i]] + part[i + 1:]

def min_inc_partition(nums, target):
    t = len(target)
    best = None
    for part in set_partitions(list(range(t))):
        Ls = [lcm(*(target[j] for j in g)) for g in part]
        tops = [heapq.nsmallest(t, ((cost(x, L), i) for i, x in enumerate(nums))) for L in Ls]
        for combo in itertools.product(*tops):
            idx = [i for _, i in combo]
            if len(set(idx)) == len(idx):
                c = sum(v for v, _ in combo)
                if best is None or c < best:
                    best = c
    return best

print('số phân hoạch của 4 phần tử:', sum(1 for _ in set_partitions([0, 1, 2, 3])))
for nums, target, exp in examples:
    print(min_inc_partition(nums, target), '| kỳ vọng', exp)

số phân hoạch của 4 phần tử: 15
1 | kỳ vọng 1
2 | kỳ vọng 2
0 | kỳ vọng 0


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2: sau mỗi phần tử in bảng `dp`.

In [10]:
def trace_dp(nums, target):
    t = len(target); full = (1 << t) - 1
    L = subset_lcms(target)
    print('BCNN theo mặt nạ:', {bin(m): L[m] for m in range(1, 1 << t)})
    INF = float('inf')
    dp = [INF] * (1 << t); dp[0] = 0
    for x in nums:
        c = [cost(x, L[m]) for m in range(1 << t)]
        new = dp[:]
        for mask in range(1 << t):
            if dp[mask] == INF:
                continue
            rest = full ^ mask; sub = rest
            while sub:
                new[mask | sub] = min(new[mask | sub], dp[mask] + c[sub])
                sub = (sub - 1) & rest
        dp = new
        print(f"sau x = {x}: chi phí theo nhóm {c[1:]} -> dp = {dp}")
    print('đáp án:', dp[full])

trace_dp([8, 4], [10, 5])

BCNN theo mặt nạ: {'0b1': 10, '0b10': 5, '0b11': 10}
sau x = 8: chi phí theo nhóm [2, 2, 2] -> dp = [0, 2, 2, 2]
sau x = 4: chi phí theo nhóm [6, 1, 6] -> dp = [0, 2, 1, 2]
đáp án: 2


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def minimumIncrements(self, nums: List[int], target: List[int]) -> int:
        t = len(target)
        full = (1 << t) - 1
        L = [1] * (1 << t)
        for m in range(1, 1 << t):
            low = (m & -m).bit_length() - 1
            L[m] = lcm(L[m ^ (1 << low)], target[low])
        keep = set()
        for m in range(1, 1 << t):
            Lm = L[m]
            for _, i in heapq.nsmallest(t, (((-x) % Lm, i) for i, x in enumerate(nums))):
                keep.add(i)
        INF = float('inf')
        dp = [INF] * (1 << t)
        dp[0] = 0
        for i in keep:
            x = nums[i]
            c = [(-x) % L[m] for m in range(1 << t)]
            new = dp[:]
            for mask in range(1 << t):
                if dp[mask] == INF:
                    continue
                rest = full ^ mask
                sub = rest
                while sub:
                    v = dp[mask] + c[sub]
                    if v < new[mask | sub]:
                        new[mask | sub] = v
                    sub = (sub - 1) & rest
            dp = new
        return dp[full]

for nums, target, exp in examples:
    got = Solution().minimumIncrements(nums, target)
    print(got, 'OK' if got == exp else 'SAI')

1 OK
2 OK
0 OK


## 10. Vì sao thuật toán đúng

1. **Cấu trúc lời giải:** mỗi phần tử `x` được dùng phục vụ một nhóm `G_x` (có thể rỗng); các nhóm phân hoạch tập target. Chi phí nhỏ nhất cho `x` với nhóm `G` là `cost(x, lcm(G))`.
2. **DP:** sau khi xét các phần tử đầu, `dp[mask]` là min chi phí phủ `mask` (mỗi phần tử chọn đúng một nhóm con của phần chưa phủ hoặc không chọn) — quy nạp chuẩn của DP "knapsack theo tập con". Việc một phần tử phục vụ target đã được phủ là thừa nên chỉ cần chọn trong phần chưa phủ.
3. **Cắt tỉa:** cố định lời giải tối ưu với các nhóm `G_1..G_r` (`r ≤ t`) và phần tử `x_1..x_r`. Nếu `x_j` không thuộc `t` phần tử rẻ nhất của `G_j`, thì trong `t` phần tử rẻ nhất đó, nhiều nhất `r − 1 ≤ t − 1` cái bị các nhóm khác dùng ⇒ còn ít nhất một cái tự do, rẻ không kém `x_j` ⇒ thay được. Lặp lại ⇒ có lời giải tối ưu chỉ dùng ứng viên.

## 11. Phân tích Time / Space Complexity

Gọi `t ≤ 4`, `n ≤ 5·10^4`.

| Cách | Time | Space |
|---|---|---|
| 1. Gán | `O(n^t · t)` | `O(t)` |
| 2. DP trên mọi `nums` | `O(n · 3^t)` ≈ `4·10^6` | `O(2^t)` |
| 3. Cắt tỉa + DP | `O(n · 2^t · log t + 60 · 3^t)` | `O(2^t · t)` |
| 4. Phân hoạch + ghép | `O(B(t) · (n log t + t^t))` | `O(t²)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'dp': min_inc_dp, 'pruned': min_inc_pruned, 'partition': min_inc_partition,
         'submit': lambda a, b: Solution().minimumIncrements(a, b)}
edge = [
    ([1], [1], 0),
    ([8, 1], [3, 5], 4),                       # không được dùng một phần tử cho hai giá trị khác nhau
    ([1, 1, 1, 1], [2, 3, 5, 7], 13),
    ([1, 1, 1, 1], [2, 3, 4, 6], 8),          # 1 -> 4 (phục vụ 2, 4) và 1 -> 6 (phục vụ 3, 6)
    ([10000], [10000], 0),
    ([1, 2], [6, 4], 7),
]
for nums, target, exp in examples + edge:
    b = min_inc_brute(nums, target)
    assert b == exp, (nums, target, b)
    for name, f in impls.items():
        assert f(nums, target) == exp, (name, nums, target)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
rng = random.Random(3444)
cnt = 0
for _ in range(1500):
    t = rng.randint(1, 4)
    n = rng.randint(t, 6)
    nums = [rng.randint(1, 40) for _ in range(n)]
    target = [rng.randint(1, 15) for _ in range(t)]
    exp = min_inc_brute(nums, target)
    for name, f in impls.items():
        assert f(nums, target) == exp, (name, nums, target)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 1500 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 5·10^4`, `t = 4`

In [14]:
import time
n = 5 * 10 ** 4
cases = {
    'ngẫu nhiên': ([rng.randint(1, 10 ** 4) for _ in range(n)], [rng.randint(1, 10 ** 4) for _ in range(4)]),
    'nums nhỏ, target lớn nguyên tố': ([rng.randint(1, 50) for _ in range(n)], [9973, 9967, 9949, 9941]),
    'toàn 1': ([1] * n, [10 ** 4, 9999, 9998, 9997]),
}
for name, (nums, target) in cases.items():
    t0 = time.time(); a = Solution().minimumIncrements(nums, target); t1 = time.time() - t0
    t0 = time.time(); b = min_inc_dp(nums, target); t2 = time.time() - t0
    assert a == b
    print(f"{name:32s}: {a:8d} | cắt tỉa {t1:.2f}s | DP toàn bộ {t2:.2f}s")

ngẫu nhiên                      :        0 | cắt tỉa 0.06s | DP toàn bộ 0.33s


nums nhỏ, target lớn nguyên tố  :    39630 | cắt tỉa 0.06s | DP toàn bộ 0.27s


toàn 1                          :    39990 | cắt tỉa 0.05s | DP toàn bộ 0.34s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định):

In [15]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        t = rng.randint(1, 3); n = rng.randint(t, 4)
        nums = [rng.randint(1, 20) for _ in range(n)]
        target = [rng.randint(1, 10) for _ in range(t)]
        if wrong(nums, target) != min_inc_brute(nums, target):
            if best is None or len(nums) + len(target) < len(best[0]) + len(best[1]):
                best = (nums, target)
    return best

### Lỗi 1 — Mỗi target tự chọn phần tử rẻ nhất (cho phép dùng một phần tử cho hai giá trị khác nhau)

In [16]:
def wrong_independent(nums, target):
    return sum(min(cost(x, b) for x in nums) for b in target)

print('sai :', wrong_independent([8, 1], [3, 5]), '| đúng:', Solution().minimumIncrements([8, 1], [3, 5]))

sai : 3 | đúng: 4


**Vì sao sai:** `8 → 9` (cho 3) và `8 → 10` (cho 5) là **hai giá trị khác nhau** của cùng một phần tử — không thể đồng thời. Muốn một phần tử phục vụ cả hai, phải lên bội của `lcm(3, 5) = 15`.

### Lỗi 2 — Dùng **tích** thay cho BCNN

In [17]:
def make_dp_with(L_fn):
    def solve(nums, target):
        t = len(target); full = (1 << t) - 1
        L = [1] * (1 << t)
        for m in range(1, 1 << t):
            low = (m & -m).bit_length() - 1
            L[m] = L_fn(L[m ^ (1 << low)], target[low])
        INF = float('inf'); dp = [INF] * (1 << t); dp[0] = 0
        for x in nums:
            new = dp[:]
            for mask in range(1 << t):
                if dp[mask] == INF: continue
                rest = full ^ mask; sub = rest
                while sub:
                    new[mask | sub] = min(new[mask | sub], dp[mask] + cost(x, L[sub]))
                    sub = (sub - 1) & rest
            dp = new
        return dp[full]
    return solve

wrong_product = make_dp_with(lambda a, b: a * b)
ce = find_counterexample(wrong_product, seed=2)
print('phản ví dụ:', ce, '| sai:', wrong_product(*ce), '| đúng:', min_inc_brute(*ce))

phản ví dụ: ([5, 17], [9, 6]) | sai: 2 | đúng: 1


**Vì sao sai:** bội chung nhỏ nhất của `4` và `6` là `12`, không phải `24`.

### Lỗi 3 — Công thức chi phí `L − x % L` (sai khi `x` đã là bội)

In [18]:
def wrong_cost(nums, target):
    best = None
    for assign in itertools.product(range(len(nums)), repeat=len(target)):
        groups = {}
        for j, i in enumerate(assign):
            groups.setdefault(i, []).append(target[j])
        c = 0
        for i, g in groups.items():
            L = lcm(*g)
            c += L - nums[i] % L                     # SAI: phải là (-x) % L
        best = c if best is None else min(best, c)
    return best

print('sai :', wrong_cost([7, 9, 10], [7]), '| đúng:', Solution().minimumIncrements([7, 9, 10], [7]))

sai : 4 | đúng: 0


**Vì sao sai:** `x = 7`, `L = 7`: `7 − 0 = 7` trong khi chi phí đúng là `0`.

### Lỗi 4 — Tham lam: xử lý target theo thứ tự giảm dần, mỗi target lấy phần tử chưa dùng rẻ nhất

In [19]:
def wrong_greedy(nums, target):
    used, total = set(), 0
    covered = set()
    for j in sorted(range(len(target)), key=lambda j: -target[j]):
        if j in covered:
            continue
        i = min((i for i in range(len(nums)) if i not in used), key=lambda i: cost(nums[i], target[j]))
        used.add(i)
        v = nums[i] + cost(nums[i], target[j])
        total += cost(nums[i], target[j])
        covered |= {jj for jj in range(len(target)) if v % target[jj] == 0}
    return total

ce = find_counterexample(wrong_greedy, seed=4)
print('phản ví dụ:', ce, '| sai:', wrong_greedy(*ce), '| đúng:', min_inc_brute(*ce))

phản ví dụ: ([4, 17], [5, 7]) | sai: 6 | đúng: 5


**Vì sao sai:** lựa chọn rẻ nhất cục bộ có thể "chiếm" phần tử mà target khác cần hơn, hoặc bỏ lỡ cơ hội gộp nhóm. Cần DP xét mọi cách gộp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Tập "yêu cầu" rất nhỏ (≤ 4–15 phần tử) ⇒ **bitmask DP** trên tập yêu cầu, duyệt phần tử như knapsack.
- "Một phần tử đáp ứng nhiều yêu cầu chia hết" ⇒ **BCNN** của nhóm.
- Duyệt tập con của phần bù ⇒ tổng `3^t`.
- Lời giải dùng ít phần tử ⇒ cắt tỉa còn top-`t` ứng viên cho mỗi nhóm.

## 15. Biến thể của bài

1. Cho phép **giảm** phần tử (không âm) ⇒ chi phí `min((−x) mod L, x mod L)`.
2. Mỗi phần tử có **trọng số chi phí** `w_i` mỗi lần tăng ⇒ `cost = w_i · ((−x) mod L)`.
3. Yêu cầu "ước" thay vì "bội" ⇒ `x` sau khi tăng phải là ước của mọi phần tử nhóm.
4. Bài liên quan: LC 1494, LC 1723 (bitmask DP gán việc).

In [20]:
# Biến thể 1: được tăng hoặc giảm (giá trị phải >= 1)
def min_moves_both_ways(nums, target):
    t = len(target); full = (1 << t) - 1
    L = subset_lcms(target)
    def c2(x, Lm):
        down = x % Lm
        return min((-x) % Lm, down if x - down >= 1 else float('inf'))
    INF = float('inf'); dp = [INF] * (1 << t); dp[0] = 0
    for x in nums:
        new = dp[:]
        for mask in range(1 << t):
            if dp[mask] == INF: continue
            rest = full ^ mask; sub = rest
            while sub:
                new[mask | sub] = min(new[mask | sub], dp[mask] + c2(x, L[sub]))
                sub = (sub - 1) & rest
        dp = new
    return dp[full]

def both_ways_brute(nums, target):
    best = None
    for assign in itertools.product(range(len(nums)), repeat=len(target)):
        groups = {}
        for j, i in enumerate(assign):
            groups.setdefault(i, []).append(target[j])
        c = 0
        for i, g in groups.items():
            Lg = lcm(*g)
            c += min(abs(v - nums[i]) for v in range(Lg, 2 * nums[i] + 2 * Lg + 1, Lg))
        best = c if best is None or c < best else best
    return best

rng = random.Random(15)
for _ in range(300):
    t = rng.randint(1, 3); n = rng.randint(t, 4)
    nums = [rng.randint(1, 30) for _ in range(n)]; target = [rng.randint(1, 10) for _ in range(t)]
    assert min_moves_both_ways(nums, target) == both_ways_brute(nums, target), (nums, target)
print('Biến thể tăng/giảm khớp vét cạn:', min_moves_both_ways([8, 4], [10, 5]), min_moves_both_ways([13], [6]))

Biến thể tăng/giảm khớp vét cạn: 2 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Gán từng target | `O(n^t)` | Oracle |
| 2 | DP bitmask trên mọi `nums` | `O(81 n)` | Đơn giản |
| 3 | Cắt tỉa top-`t` + DP | `O(15 n log t)` | **Nộp bài** |
| 4 | Phân hoạch + ghép | `O(15 n)` | Góc nhìn tổ hợp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [5, 5]`, `target = [2, 3]`.

**Bài 2 (tính tay):** `nums = [6]`, `target = [4, 6]`.

**Bài 3 (điền TODO):** viết `weighted(nums, w, target)` — biến thể 2.

**Bài 4 (tự viết lại):** trả về **giá trị mới** của mỗi phần tử trong một lời giải tối ưu.

**Bài 5 (mini challenge):** chứng minh bằng thực nghiệm rằng giữ **`t − 1`** ứng viên mỗi nhóm là **không đủ** (tìm phản ví dụ).

In [21]:
def weighted_todo(nums, w, target):
    # TODO
    raise NotImplementedError

def reconstruct_todo(nums, target):
    # TODO
    raise NotImplementedError

def find_prune_counterexample_todo():
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Riêng: `5 → 6` (1) cho 2 hoặc 3? `6` chia hết cả 2 và 3! (2) `lcm = 6`. (3) Đáp án 1.

**Bài 2.** (1) Chỉ một phần tử ⇒ phải phục vụ cả hai. (2) `lcm(4, 6) = 12`. (3) `6 → 12`: 6.

**Bài 3.** (1) Nhân chi phí với `w_i`. (2) Cắt tỉa theo chi phí có trọng số. (3) Đối chiếu vét cạn.

**Bài 4.** (1) Lưu vết `(mask cũ, sub, i)` trong DP. (2) Truy ngược từ `full`. (3) Kiểm tra mọi target có bội trong mảng mới.

**Bài 5.** (1) Thay `t` bằng `t − 1` trong `nsmallest`. (2) So với vét cạn trên nhiều bộ nhỏ. (3) Phản ví dụ cần nhiều nhóm cùng "tranh" một phần tử rẻ.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
a1 = Solution().minimumIncrements([5, 5], [2, 3])
a2 = Solution().minimumIncrements([6], [4, 6])
assert a1 == min_inc_brute([5, 5], [2, 3]) == 1 and a2 == min_inc_brute([6], [4, 6]) == 6
print('Bài 1:', a1, '| Bài 2:', a2)

# Bài 3
def weighted(nums, w, target):
    t = len(target); full = (1 << t) - 1
    L = subset_lcms(target)
    INF = float('inf'); dp = [INF] * (1 << t); dp[0] = 0
    for x, wi in zip(nums, w):
        new = dp[:]
        for mask in range(1 << t):
            if dp[mask] == INF: continue
            rest = full ^ mask; sub = rest
            while sub:
                new[mask | sub] = min(new[mask | sub], dp[mask] + wi * cost(x, L[sub]))
                sub = (sub - 1) & rest
        dp = new
    return dp[full]

def weighted_brute(nums, w, target):
    best = None
    for assign in itertools.product(range(len(nums)), repeat=len(target)):
        groups = {}
        for j, i in enumerate(assign):
            groups.setdefault(i, []).append(target[j])
        c = sum(w[i] * cost(nums[i], lcm(*g)) for i, g in groups.items())
        best = c if best is None or c < best else best
    return best

rng = random.Random(3)
for _ in range(400):
    t = rng.randint(1, 3); n = rng.randint(t, 4)
    nums = [rng.randint(1, 20) for _ in range(n)]; w = [rng.randint(1, 5) for _ in range(n)]
    target = [rng.randint(1, 10) for _ in range(t)]
    assert weighted(nums, w, target) == weighted_brute(nums, w, target)
print('Bài 3 khớp vét cạn.')

Bài 1: 1 | Bài 2: 6
Bài 3 khớp vét cạn.


In [23]:
# Bài 4, 5
def reconstruct(nums, target):
    t = len(target); full = (1 << t) - 1
    L = subset_lcms(target)
    INF = float('inf')
    layers = [[INF] * (1 << t)]
    layers[0][0] = 0
    choice = []
    for x in nums:
        dp = layers[-1]; new = dp[:]; ch = [None] * (1 << t)
        for mask in range(1 << t):
            if dp[mask] == INF: continue
            rest = full ^ mask; sub = rest
            while sub:
                v = dp[mask] + cost(x, L[sub])
                if v < new[mask | sub]:
                    new[mask | sub] = v; ch[mask | sub] = (mask, sub)
                sub = (sub - 1) & rest
        layers.append(new); choice.append(ch)
    out, mask = list(nums), full
    for i in range(len(nums) - 1, -1, -1):
        if layers[i + 1][mask] == layers[i][mask]:
            continue
        prev, sub = choice[i][mask]
        out[i] = nums[i] + cost(nums[i], L[sub])
        mask = prev
    return layers[-1][full], out

rng = random.Random(4)
for _ in range(300):
    t = rng.randint(1, 4); n = rng.randint(t, 6)
    nums = [rng.randint(1, 30) for _ in range(n)]; target = [rng.randint(1, 12) for _ in range(t)]
    c, new = reconstruct(nums, target)
    assert c == min_inc_brute(nums, target) and sum(b - a for a, b in zip(nums, new)) == c
    assert all(any(v % b == 0 for v in new) for b in target)
print('Bài 4: dựng lại hợp lệ; ví dụ 2 ->', reconstruct([8, 4], [10, 5]))

def pruned_k(nums, target, keep_k):
    t = len(target); L = subset_lcms(target)
    keep = set()
    for m in range(1, 1 << t):
        for _, i in heapq.nsmallest(keep_k, ((cost(x, L[m]), i) for i, x in enumerate(nums))):
            keep.add(i)
    return min_inc_dp([nums[i] for i in sorted(keep)], target)

found = None
rng = random.Random(55)
for _ in range(20000):
    t = rng.randint(2, 4); n = rng.randint(t, 7)
    nums = [rng.randint(1, 12) for _ in range(n)]; target = [rng.randint(1, 8) for _ in range(t)]
    if pruned_k(nums, target, t - 1) != min_inc_brute(nums, target):
        found = (nums, target); break
print('Bài 5: phản ví dụ khi chỉ giữ t-1 ứng viên:', found,
      '->', pruned_k(*found, len(found[1]) - 1), 'thay vì', min_inc_brute(*found))

Bài 4: dựng lại hợp lệ; ví dụ 2 -> (2, [10, 4])
Bài 5: phản ví dụ khi chỉ giữ t-1 ứng viên: ([11, 2, 8], [8, 5]) -> 4 thay vì 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
L[mask] = lcm các target trong mask
cand = ∪_mask top-t phần tử theo (-x) % L[mask]
dp[0] = 0
for x in cand:
    new = dp[:]
    for mask: for sub ⊆ full ^ mask, sub ≠ 0:
        new[mask|sub] = min(new[mask|sub], dp[mask] + (-x) % L[sub])
    dp = new
return dp[full]
```

**Checklist:**

- [ ] Chi phí `(−x) % L` (bằng 0 khi đã là bội)?
- [ ] BCNN, không phải tích?
- [ ] Một phần tử chỉ nhận **một** giá trị mới (phục vụ cả nhóm qua BCNN)?
- [ ] Cắt tỉa giữ đủ `t` ứng viên mỗi nhóm?
- [ ] Dùng bản sao `dp` (mỗi phần tử dùng tối đa một lần)?